In [1]:
import time
import torch
import numpy as np

import lightning.pytorch as pl
import matplotlib.pyplot as plt

from data import DatasetsConfig, SLRDatasetOLS
from models import ModelConfig, FinancialLSTM_MGNLL, MultivariateGaussianNLLLoss
from torch.utils.data import DataLoader, ConcatDataset

from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import (
    ModelCheckpoint, 
    LearningRateMonitor
)

# Data
data.py

In [2]:
datasets_cfg = DatasetsConfig(
    n_stock        = 10,
    n_windows      = 1000,
    lookback_window= 60,
    target_window  = 20,
    prediction_task= True,
    n_dataset_train= 70,
    n_dataset_val  = 20,
    n_dataset_test = 10
)

datasets = SLRDatasetOLS.generate_datasets(datasets_cfg)

# Model
models.py

In [86]:
model_cfg = ModelConfig(
    hidden_size    = 64,
    num_layers     = 2,
    dropout        = 0.2,
    learning_rate  = 1e-4,
    num_epochs     = 32,
    batch_size     = 1,
    shuffle_batches= True,
    lr_patience    = 4,
    loss_fn        = MultivariateGaussianNLLLoss(
        n = datasets_cfg.target_window,
        K = datasets_cfg.n_stock
    )
)

model = FinancialLSTM_MGNLL(model_cfg)

## Config

In [83]:
PATH = [
    'logs/financial_lstm', # Save dir
    'v8',                  # Name
    'prediction_run_MGNLL=N_TRAIN=70__BATCH=1_EPOCH=32' # Version
]

## Datasets and DataLoaders

In [5]:
train_datasets, val_datasets, test_datasets = datasets

train_loader = DataLoader(ConcatDataset(train_datasets), batch_size=model_cfg.batch_size, shuffle=model_cfg.shuffle_batches, num_workers=8)
val_loader   = DataLoader(ConcatDataset(val_datasets), batch_size=model_cfg.batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(ConcatDataset(test_datasets),  batch_size=1, shuffle=False)

## Trainer

#### Logger

In [84]:
tb_logger = TensorBoardLogger(
    save_dir          = PATH[0],
    name              = PATH[1],
    version           = PATH[2],
    default_hp_metric = True
)

#### Callbacks

In [85]:
callbacks = [
    ModelCheckpoint(
        dirpath=f"{PATH[0]}/{PATH[1]}/{PATH[2]}/checkpoints",
        filename="best_{epoch:02d}_{val_loss:.6f}",
        monitor="val_loss",
        mode="min",
        save_last=True,
        auto_insert_metric_name=True
    ),
    
    LearningRateMonitor(
        logging_interval="epoch",
        log_momentum=False
    )
]

#### Trainer

In [94]:
trainer = pl.Trainer(
    max_epochs = model_cfg.num_epochs,
    logger     = tb_logger,
    callbacks  = callbacks,
    accelerator='cpu',
    devices='auto',
    log_every_n_steps=10,
    enable_progress_bar=True,
    enable_model_summary=True,
    gradient_clip_val=1.0
)

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/marko/Environments/python/fer/dipl/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:166: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


## Train model

In [ ]:
# ==================== Train the Model ====================
print("\n" + "="*50)
print("Starting Training...")
print("="*50 + "\n")

trainer.fit(model, train_loader, val_loader)

# ==================== Print Results ====================
print("\n" + "="*50)
print("Training Complete!")
print("="*50)
print(f"Best model path: {trainer.checkpoint_callback.best_model_path}")
print(f"Best validation loss: {trainer.checkpoint_callback.best_model_score:.6f}")
print("="*50 + "\n")

## Resume training

In [ ]:
# TODO Test if it works

model = FinancialLSTM_MGNLL(model_cfg)
#trainer = pl.Trainer()

# ==================== Train the Model ====================
print("\n" + "="*50)
print("Resuming Training...")
print("="*50 + "\n")

trainer.fit(model, train_loader, val_loader, ckpt_path='logs/financial_lstm/v8/prediction_run_MGNLL=N_TRAIN=70__BATCH=1_EPOCH=16/checkpoints/last.ckpt')

# ==================== Print Results ====================
print("\n" + "="*50)
print("Training Complete!")
print("="*50)
print(f"Best model path: {trainer.checkpoint_callback.best_model_path}")
print(f"Best validation loss: {trainer.checkpoint_callback.best_model_score:.6f}")
print("="*50 + "\n")

Restoring states from the checkpoint path at logs/financial_lstm/v8/prediction_run_MGNLL=N_TRAIN=70__BATCH=1_EPOCH=16/checkpoints/last.ckpt
/home/marko/Environments/python/fer/dipl/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:445: The dirpath has changed from '/home/marko/Workspace/FER/masters-thesis/logs/financial_lstm/v8/prediction_run_MGNLL=N_TRAIN=70__BATCH=1_EPOCH=16/checkpoints' to '/home/marko/Workspace/FER/masters-thesis/logs/financial_lstm/v8/prediction_run_MGNLL=N_TRAIN=70__BATCH=1_EPOCH=32/checkpoints', therefore `best_model_score`, `kth_best_model_path`, `kth_value`, `last_model_path` and `best_k_models` won't be reloaded. Only `best_model_path` will be reloaded.

  | Name         | Type    | Params | Mode 
-------------------------------------------------
0 | lstm         | LSTM    | 50.9 K | train
1 | alpha_head   | Linear  | 65     | train
2 | beta_head    | Linear  | 65     | train
3 | test_loss_fn | MSELoss | 0      | train
-------------


Resuming Training...



Sanity Checking: |                                                                                            …

Training: |                                                                                                   …

Validation: |                                                                                                 …

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

## Load model

# Benchmark

In [96]:
from data import add_intercept

true_alphas = torch.stack([test_dataset.alphas for test_dataset in test_datasets])
true_betas  = torch.stack([test_dataset.betas for test_dataset in test_datasets])

loss_function = torch.nn.MSELoss(reduction='mean')

model_summary = {
    'recon_loss': [],
    'alpha_loss': [],
    'beta_loss':[],
    'alpha': [],
    'beta': []
}

ols_summary = {
    'recon_loss': [],
    'alpha_loss': [],
    'beta_loss':[],
    'alpha': [],
    'beta': []
}

n_stock = datasets_cfg.n_stock
len_dataset = datasets_cfg.n_windows

start_time = time.time()
print("Evauating model on test set...")

model.eval()
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        curr_dataset = i // len_dataset

        true_alpha = true_alphas[curr_dataset]
        true_beta  = true_betas[curr_dataset]

        r_context        = batch['r_context'].squeeze()
        r_market_context = batch['r_market_context'].squeeze()
        r_market_target  = batch['r_market_target'].squeeze()
        r_target         = batch['r_target'].squeeze()
        r_market_expanded = r_market_context.repeat(r_context.shape[0], 1)

        # Model loss
        alpha, beta = model(r_context, r_market_expanded)
        reconstruction = alpha + beta * r_market_target
        
        model_summary['recon_loss'].append(loss_function(reconstruction, r_target).item())
        model_summary['alpha_loss'].append(loss_function(alpha, true_alpha).item())
        model_summary['beta_loss'].append(loss_function(beta, true_beta).item())

        model_summary['alpha'].append(alpha.detach())
        model_summary['beta'].append(beta.detach())

        
        #OLS and OLS loss
        X = r_market_context
        Y = r_context.T
        
        X = add_intercept(X)
        
        # OLS
        OLS = torch.linalg.pinv(X.T @ X) @ (X.T @ Y)
        alpha = OLS[0, :].unsqueeze(1)
        beta  = OLS[1, :].unsqueeze(1)
        reconstruction = alpha.unsqueeze(-1) + beta.unsqueeze(-1) * r_market_target

        ols_summary['recon_loss'].append(loss_function(reconstruction, r_target).item())
        ols_summary['alpha_loss'].append(loss_function(alpha, true_alpha).item())
        ols_summary['beta_loss'].append(loss_function(beta, true_beta).item())

        ols_summary['alpha'].append(alpha)
        ols_summary['beta'].append(beta)

"""
model_summary['alpha'] = torch.Tensor(model_summary['alpha']).reshape(datasets_cfg.n_dataset_test, datasets_cfg.n_windows, datasets_cfg.n_stock)
model_summary['beta'] = torch.Tensor(model_summary['beta']).reshape(datasets_cfg.n_dataset_test, datasets_cfg.n_windows, datasets_cfg.n_stock)

ols_summary['alpha'] = torch.Tensor(ols_summary['alpha']).reshape(datasets_cfg.n_dataset_test, datasets_cfg.n_windows, datasets_cfg.n_stock)
ols_summary['beta'] = torch.Tensor(ols_summary['beta']).reshape(datasets_cfg.n_dataset_test, datasets_cfg.n_windows, datasets_cfg.n_stock)
"""

print(f"Done! time = {time.time() - start_time:>7f} seconds")

Evauating model on test set...


/home/marko/Environments/python/fer/dipl/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([10])) that is different to the input size (torch.Size([10, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/marko/Environments/python/fer/dipl/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([10, 20])) that is different to the input size (torch.Size([10, 1, 20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Done! time = 4.504267 seconds


In [97]:
# Calculate test metrics for hp_metric
model_test_loss = np.mean(model_summary['recon_loss'])
ols_test_loss = np.mean(ols_summary['recon_loss'])

# Log hp_metric (model_test_loss)
tb_logger.log_hyperparams(
    params=asdict(model_cfg),
    metrics={'hp_metric': model_test_loss}
)

In [98]:
plt.figure(figsize=(10, 8))

plt.scatter(model_summary['recon_loss'], ols_summary['recon_loss'], marker='.')
plt.xlabel('Model Loss')
plt.ylabel('OLS Loss')

identity = np.linspace(min(ols_summary['recon_loss']), max(ols_summary['recon_loss']), 1000)
plt.plot(identity, identity, 'r--')

plt.title('Model vs OLS Reconstruction loss (MSE)')
plt.grid(alpha=0.5)

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Reconstruction loss (MSE)', plt.gcf())

In [99]:
plt.figure(figsize=(10, 8))

plt.hist(model_summary['recon_loss'], bins=100, alpha=0.6, label='Model')
plt.hist(ols_summary['recon_loss'], bins=100, alpha=0.6, label='OLS')

plt.axvline(model_test_loss, color='blue', linestyle='--', label=f'Avg Model Loss ({model_test_loss:.7f})')
plt.axvline(ols_test_loss, color='orange', linestyle='--', label=f'Avg OLS Loss    ({ols_test_loss:.7f})')

plt.title('Model vs OLS Reconstruction loss (MSE)')
plt.grid(alpha=0.5)
plt.legend()

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Reconstruction loss (MSE) Histogram', plt.gcf())

In [100]:
model_alpha_test_loss = np.mean(model_summary['alpha_loss'])
ols_alpha_test_loss = np.mean(ols_summary['alpha_loss'])

plt.figure(figsize=(10, 8))

plt.hist(model_summary['alpha_loss'], bins=100, alpha=0.6, label='Model')
plt.hist(ols_summary['alpha_loss'], bins=100, alpha=0.6, label='OLS')

plt.axvline(model_alpha_test_loss, color='blue', linestyle='--', label=f'Avg Model Loss ({model_alpha_test_loss:.7f})')
plt.axvline(ols_alpha_test_loss, color='orange', linestyle='--', label=f'Avg OLS Loss    ({ols_alpha_test_loss:.7f})')

plt.title('Model vs OLS Alpha loss (MSE)')
plt.grid(alpha=0.5)
plt.legend()

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Alpha loss (MSE)', plt.gcf())

In [101]:
model_beta_test_loss = np.mean(model_summary['beta_loss'])
ols_beta_test_loss = np.mean(ols_summary['beta_loss'])

plt.figure(figsize=(10, 8))

plt.hist(model_summary['beta_loss'], bins=100, alpha=0.6, label='Model')
plt.hist(ols_summary['beta_loss'], bins=100, alpha=0.6, label='OLS')

plt.axvline(model_beta_test_loss, color='blue', linestyle='--', label=f'Avg Model Loss ({model_beta_test_loss:.4f})')
plt.axvline(ols_beta_test_loss, color='orange', linestyle='--', label=f'Avg OLS Loss    ({ols_beta_test_loss:.4f})')

plt.title('Model vs OLS Beta loss (MSE)')
plt.grid(alpha=0.5)
plt.legend()

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Beta loss (MSE)', plt.gcf())

In [102]:
idxs = list(range(9))

model_summary['alpha'] = torch.stack(model_summary['alpha']).reshape(datasets_cfg.n_dataset_test, datasets_cfg.n_windows, datasets_cfg.n_stock)
ols_summary['alpha'] = torch.stack(ols_summary['alpha']).reshape(datasets_cfg.n_dataset_test, datasets_cfg.n_windows, datasets_cfg.n_stock)

model_summary['beta'] = torch.stack(model_summary['beta']).reshape(datasets_cfg.n_dataset_test, datasets_cfg.n_windows, datasets_cfg.n_stock)
ols_summary['beta'] = torch.stack(ols_summary['beta']).reshape(datasets_cfg.n_dataset_test, datasets_cfg.n_windows, datasets_cfg.n_stock)

In [103]:
fig, axs = plt.subplots(3, 3, figsize=(10, 8), sharex=True, sharey=True)
fig.suptitle('Model vs OLS Alpha estimation')

axs = axs.flatten()

for ax, idx in zip(axs, idxs):
    dataset_idx = idx
    
    true_alpha  = true_alphas[dataset_idx, 0]
    model_alpha = model_summary['alpha'][dataset_idx, :, 0]
    ols_alpha   = ols_summary['alpha'][dataset_idx, :, 0]

    sample = np.arange(len(model_alpha))

    ax.set_title(f'Dataset {idx}, Stock 0')
    ax.scatter(sample, model_alpha, label='Model', marker='.')
    ax.scatter(sample, ols_alpha, label='OLS', marker='.')
    ax.axhline(true_alpha, color='magenta', linestyle='--', label='True alpha')

    ax.legend()
    ax.grid(alpha=0.5)

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Alpha estimation', fig)

In [104]:
fig, axs = plt.subplots(3, 3, figsize=(10, 8), sharex=True, sharey=True)
fig.suptitle('Model vs OLS Beta estimation')

axs = axs.flatten()

for ax, idx in zip(axs, idxs):
    dataset_idx = idx
    
    true_beta  = true_betas[dataset_idx, 0]
    model_beta = model_summary['beta'][dataset_idx, :, 0]
    ols_beta   = ols_summary['beta'][dataset_idx, :, 0]

    sample = np.arange(len(model_beta))

    ax.set_title(f'Dataset {idx}, Stock 0')
    ax.scatter(sample, model_beta, label='Model', marker='.')
    ax.scatter(sample, ols_beta, label='OLS', marker='.')
    ax.axhline(true_beta, color='magenta', linestyle='--', label='True beta')

    ax.legend()
    ax.grid(alpha=0.5)

# Log to TensorBoard
tb_logger.experiment.add_figure('Model vs OLS Beta estimation', fig)

In [105]:
fig = plt.figure(figsize=(10, 8))
plt.hist(model_summary['beta'].flatten(), bins=500)
plt.title('Model beta distribution')

plt.grid()
tb_logger.experiment.add_figure('Model Beta Distribution', fig)

In [ ]:
idxs = list(range(datasets_cfg.n_stock * datasets_cfg.n_dataset_test))
n = datasets_cfg.n_windows

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, sharey=True)
fig.suptitle('Ground Truth Beta vs Estimated Beta')

identity = np.linspace(true_betas.min(), true_betas.max(), 1000)

# Top subplot - Model
ax1.set_title('LSTM Model')
ax1.set_ylabel('Model Beta')
ax1.plot(identity, identity, color='magenta', linestyle='--')
ax1.grid()

# Bottom subplot - OLS
ax2.set_title('OLS')
ax2.set_xlabel('Ground Truth Beta')
ax2.set_ylabel('OLS Beta ')
ax2.plot(identity, identity, color='magenta', linestyle='--')
ax2.grid()

for idx in idxs:
    stock_idx   = idx % datasets_cfg.n_stock
    dataset_idx = idx // n
    
    true_beta  = true_betas[dataset_idx, stock_idx]
    model_beta = model_summary['beta'][dataset_idx, :, stock_idx]
    ols_beta   = ols_summary['beta'][dataset_idx, :, stock_idx]

    x = [true_beta] * n

    ax1.scatter(x, model_beta, marker='.', alpha=0.1)
    ax2.scatter(x, ols_beta, marker='.', alpha=0.1)

tb_logger.experiment.add_figure('Ground Truth Beta vs Estimated Beta', fig)

In [110]:
idxs = list(range(datasets_cfg.n_stock * datasets_cfg.n_dataset_test))
n = datasets_cfg.n_windows

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, sharey=True)
fig.suptitle('Ground Truth Alpha vs Estimated Alpha')

identity = np.linspace(true_alphas.min(), true_alphas.max(), 1000)

# Top subplot - Model
ax1.set_title('LSTM Model')
ax1.set_ylabel('Model Alpha')
ax1.plot(identity, identity, color='magenta', linestyle='--')
ax1.grid()

# Bottom subplot - OLS
ax2.set_title('OLS')
ax2.set_xlabel('Ground Truth Alpha')
ax2.set_ylabel('OLS Alpha ')
ax2.plot(identity, identity, color='magenta', linestyle='--')
ax2.grid()

for idx in idxs:
    stock_idx   = idx % datasets_cfg.n_stock
    dataset_idx = idx // n
    
    true_alpha  = true_alphas[dataset_idx, stock_idx]
    model_alpha = model_summary['alpha'][dataset_idx, :, stock_idx]
    ols_alpha   = ols_summary['alpha'][dataset_idx, :, stock_idx]

    x = [true_alpha] * n

    ax1.scatter(x, model_alpha, marker='.', alpha=0.1)
    ax2.scatter(x, ols_alpha, marker='.', alpha=0.1)

tb_logger.experiment.add_figure('Ground Truth Alpha vs Estimated Alpha', fig)

In [111]:
tb_logger.save()